In [ ]:
import os
import pickle

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from scipy.sparse import csr_matrix
from itertools import combinations
from sklearn.model_selection import train_test_split
import re
from collections import defaultdict
from tqdm import tqdm
tqdm.pandas()

df_final = pd.read_csv('../data/df_final_prepared.csv')

In [2]:
print("ПРОВЕРКА product_id")

print("\nУникальных товаров:")
print(df_final['product_id'].nunique())

print("\nПример:")
display(df_final[['product_id', 'name_clean']].head())

ПРОВЕРКА product_id

Уникальных товаров:
27212

Пример:


,product_id,name_clean
0,0,коньяк армянский аревик 5лет 40 0 5л под уп а...
1,0,коньяк армянский аревик 5лет 40 0 5л под уп а...
2,0,коньяк армянский аревик 5лет 40 0 5л под уп а...
3,0,коньяк армянский аревик 5лет 40 0 5л под уп а...
4,0,коньяк армянский аревик 5лет 40 0 5л под уп а...


In [3]:
print("PRODUCT_ID → INDEX")

# уникальные товары
unique_products = sorted(df_final['product_id'].unique())

# mapping
product_to_idx = {p: i for i, p in enumerate(unique_products)}
idx_to_product = {i: p for p, i in product_to_idx.items()}

n_items = len(unique_products)

print(f"\nВсего товаров: {n_items}")

print("\nПример mapping:")
for i, p in enumerate(unique_products[:5]):
    print(f"{p} → {product_to_idx[p]}")

PRODUCT_ID → INDEX

Всего товаров: 27212

Пример mapping:
0 → 0
1 → 1
2 → 2
3 → 3
4 → 4


In [4]:
print("СОЗДАНИЕ КОРЗИН")

baskets = df_final.groupby('cheque_id')['product_id'].apply(list).reset_index()

print(f"\nВсего корзин: {len(baskets)}")

print("\nПример корзины:")
print(baskets.iloc[0])

СОЗДАНИЕ КОРЗИН

Всего корзин: 49337

Пример корзины:
cheque_id                                5770252090.0
product_id    [161, 5538, 12965, 21534, 21498, 26583]
Name: 0, dtype: object


In [5]:
print("SPARSE MATRIX")

rows = []
cols = []

for i, row in baskets.iterrows():
    
    if i % 10000 == 0:
        print(f"Обработано: {i}")
    
    items = row['product_id']
    
    for item in items:
        if item in product_to_idx:
            rows.append(i)
            cols.append(product_to_idx[item])

data = np.ones(len(rows), dtype=np.float32)

X = csr_matrix((data, (rows, cols)), shape=(len(baskets), n_items))

print("\nMatrix shape:")
print(X.shape)

print("\nненулевых элементов:")
print(X.nnz)

print("\nПлотность:")
print(X.nnz / (X.shape[0] * X.shape[1]))

SPARSE MATRIX
Обработано: 0
Обработано: 10000
Обработано: 20000
Обработано: 30000
Обработано: 40000

Matrix shape:
(49337, 27212)

ненулевых элементов:
768420

Плотность:
0.000572354971535228


In [6]:
print("TRAIN / TEST SPLIT")

train_idx, test_idx = train_test_split(
    np.arange(X.shape[0]),
    test_size=0.2,
    random_state=42
)

X_train = X[train_idx]
X_test  = X[test_idx]

print(f"Train: {X_train.shape}")
print(f"Test:  {X_test.shape}")

TRAIN / TEST SPLIT
Train: (39469, 27212)
Test:  (9868, 27212)


In [7]:
print("ОБУЧЕНИЕ EASE")

# параметр регуляризации
lambda_ = 500.0

print(f"lambda = {lambda_}")
print("\nСчитаем G = X^T X")

G = (X_train.T @ X_train).toarray().astype(np.float32)

print("G shape:", G.shape)

# регуляризация
diag_idx = np.diag_indices(G.shape[0])
G[diag_idx] += lambda_

# обратная матрица
print("\nИнвертируем G")

P = np.linalg.inv(G)

# финальная матрица весов
B = P / (-np.diag(P))
B[diag_idx] = 0.0

print("\nМатрица B готова")
print("Shape:", B.shape)

ОБУЧЕНИЕ EASE
lambda = 500.0

Считаем G = X^T X
G shape: (27212, 27212)

Инвертируем G

Матрица B готова
Shape: (27212, 27212)


In [9]:
print("RECALL@10 ДЛЯ EASE")

def recall_ease(B, X_test, k=10, n_samples=3000, seed=42):
    rng = np.random.default_rng(seed)
    
    indices = rng.choice(X_test.shape[0], size=n_samples, replace=False)
    
    hits = 0
    total = 0
    
    for i in tqdm(indices):
        row = X_test[i].toarray().flatten()
        
        items = np.where(row == 1)[0]
        
        if len(items) < 2:
            continue
        
        target = rng.choice(items)
        context = [x for x in items if x != target]
        
        if len(context) == 0:
            continue
        
        scores = np.zeros(B.shape[0])
        
        for c in context:
            scores += B[c]
        
        scores[context] = -np.inf
        
        top_k = np.argpartition(-scores, k)[:k]
        
        if target in top_k:
            hits += 1
        
        total += 1
    
    return hits / total if total > 0 else 0.0


ease_recall = recall_ease(B, X_test, k=10, n_samples=3000)

print(f"EASE Recall@10: {ease_recall:.6f}")

RECALL@10 ДЛЯ EASE


100%|██████████| 3000/3000 [00:02<00:00, 1250.80it/s]

EASE Recall@10: 0.114401


In [ ]:
print("TUNING lambda ДЛЯ EASE")

lambda_grid = [0.1, 0.3, 0.5, 0.7, 1.0, 10.0, 30.0, 50.0, 70.0, 100.0, 130.0, 150.0, 250.0]
ease_results = []

for lambda_ in lambda_grid:
    
    # G = X^T X
    G = (X_train.T @ X_train).toarray().astype(np.float32)
    
    # регуляризация
    diag_idx = np.diag_indices(G.shape[0])
    G[diag_idx] += lambda_
    
    # inverse
    P = np.linalg.inv(G)
    
    # EASE weights
    B = P / (-np.diag(P))
    B[diag_idx] = 0.0
    
    # recall
    ease_recall = recall_ease(B, X_test, k=10, n_samples=5000, seed=1211)
    
    ease_results.append((lambda_, ease_recall))

print("ИТОГИ TUNING")

for lambda_, score in ease_results:
    print(f"lambda={lambda_:>6.1f} | Recall@10={score:.6f}")

best_lambda, best_score = max(ease_results, key=lambda x: x[1])

print("\nЛУЧШИЙ РЕЗУЛЬТАТ:")
print(f"lambda={best_lambda:.1f}")
print(f"Recall@10={best_score:.6f}")

TUNING lambda ДЛЯ EASE
ИТОГИ TUNING
lambda=   0.1 | Recall@10=0.046330
lambda=   0.3 | Recall@10=0.064677
lambda=   0.5 | Recall@10=0.068374
lambda=   0.7 | Recall@10=0.071906
lambda=   1.0 | Recall@10=0.078348
lambda=  10.0 | Recall@10=0.102524
lambda=  30.0 | Recall@10=0.114602
lambda=  50.0 | Recall@10=0.117623
lambda=  70.0 | Recall@10=0.118630
lambda= 100.0 | Recall@10=0.120645
lambda= 130.0 | Recall@10=0.121047
lambda= 150.0 | Recall@10=0.121249
lambda= 250.0 | Recall@10=0.119104

ЛУЧШИЙ РЕЗУЛЬТАТ:
lambda=150.0
Recall@10=0.121249


In [ ]:
print("FINAL lambda ДЛЯ EASE")

lambda_ = best_lambda
print(f"lambda = {lambda_}")

# G = X^T X
G = (X_train.T @ X_train).toarray().astype(np.float32)

# регуляризация
diag_idx = np.diag_indices(G.shape[0])
G[diag_idx] += lambda_

# inverse
P = np.linalg.inv(G)

# EASE weights
B = P / (-np.diag(P))
B[diag_idx] = 0.0

ease_recall = recall_ease(B, X_test, k=10, n_samples=3000)
print(f"EASE Recall@10: {ease_recall}")

FINAL lambda ДЛЯ EASE
lambda = 150.0


100%|██████████| 3000/3000 [00:02<00:00, 1244.01it/s]

EASE Recall@10: 0.12487445597589555


In [ ]:
print("СОХРАНЕНИЕ EASE МОДЕЛИ")

save_dir = "../data/ease_final"
os.makedirs(save_dir, exist_ok=True)

# СОХРАНЯЕМ МАТРИЦУ B
np.save(f"{save_dir}/B.npy", B)

print("B сохранена")

# СОХРАНЯЕМ MAPPING
with open(f"{save_dir}/product_to_idx.pkl", "wb") as f:
    pickle.dump(product_to_idx, f)

with open(f"{save_dir}/idx_to_product.pkl", "wb") as f:
    pickle.dump(idx_to_product, f)

print("mapping сохранены")

# СОХРАНЯЕМ CONFIG
config = {
    "lambda": best_lambda,
    "recall@10": ease_recall
}

with open(f"{save_dir}/config.pkl", "wb") as f:
    pickle.dump(config, f)

print("config сохранён")

# ФИНАЛЬНЫЙ ВЫВОД
print("FINAL EASE MODEL")
print(f"Recall@10: {ease_recall:.6f}")

СОХРАНЕНИЕ EASE МОДЕЛИ
B сохранена
mapping сохранены
config сохранён
FINAL EASE MODEL
Recall@10: 0.124874
